# Advanced Problems with Solutions: Generator Expressions

This notebook expands the generator-expression topic into advanced, practice-oriented problems with complete runnable solutions.

You will practice lazy evaluation, iterator exhaustion, nested generators, short-circuiting, streaming pipelines, memory checks, infinite streams, debugging helpers, and reusable lazy ETL patterns.


## Best-Practice Checklist

- Use generator expressions when the data may be large or consumed once.
- Materialize with `list(...)` only when you truly need indexing, reuse, or display.
- Remember: generators are single-use iterators.
- Prefer readable staged pipelines over one massive expression.
- Use `next`, `any`, `all`, `sum`, `min`, `max`, `islice`, and `itertools` with generators for memory-friendly code.


In [1]:
from __future__ import annotations

from collections import Counter, deque
from itertools import chain, islice, tee, takewhile, dropwhile
from math import factorial, isclose
from pathlib import Path
from random import randint, seed
from tempfile import NamedTemporaryFile
import heapq
import sys
import tracemalloc

seed(42)


## Warm-up: List Comprehension vs Generator Expression


In [2]:
squares_list = [i ** 2 for i in range(8)]
squares_gen = (i ** 2 for i in range(8))

print(type(squares_list).__name__, squares_list)
print(type(squares_gen).__name__, squares_gen)

print(next(squares_gen), next(squares_gen), next(squares_gen))
print(list(squares_gen))
print(list(squares_gen))  # exhausted


list [0, 1, 4, 9, 16, 25, 36, 49]
generator <generator object <genexpr> at 0x0000023951402190>
0 1 4
[9, 16, 25, 36, 49]
[]


---

## Problem 1 — Exhaustion-Aware Streaming Statistics

Write `stream_stats(values)` that accepts any iterable of numbers and returns count, total, mean, minimum, and maximum.

Constraints:

- Do not convert the iterable to a list.
- Work with one-time generators.
- Use one pass only.
- Return `None` for mean/minimum/maximum when empty.


In [3]:
def stream_stats(values):
    count = 0
    total = 0
    minimum = None
    maximum = None

    for value in values:
        count += 1
        total += value
        minimum = value if minimum is None or value < minimum else minimum
        maximum = value if maximum is None or value > maximum else maximum

    return {
        "count": count,
        "total": total,
        "mean": total / count if count else None,
        "minimum": minimum,
        "maximum": maximum,
    }

assert stream_stats([]) == {
    "count": 0,
    "total": 0,
    "mean": None,
    "minimum": None,
    "maximum": None,
}

g = (x * x for x in range(6))
result = stream_stats(g)
print(result)
assert result == {"count": 6, "total": 55, "mean": 55 / 6, "minimum": 0, "maximum": 25}
assert list(g) == []  # consumed once


{'count': 6, 'total': 55, 'mean': 9.166666666666666, 'minimum': 0, 'maximum': 25}


---

## Problem 2 — Streaming Log Parser Pipeline

Given log lines, build a lazy pipeline that parses rows, removes malformed rows, keeps only `ERROR` records, extracts milliseconds, and computes the average error response time.


In [4]:
sample_logs = [
    "2026-06-30 INFO user=ana action=login ms=31",
    "2026-06-30 ERROR user=bob action=download ms=870",
    "2026-06-30 WARNING user=chen action=upload ms=120",
    "bad line that should be ignored",
    "2026-06-30 ERROR user=dina action=checkout ms=430",
    "2026-06-30 ERROR user=eli action=search ms=90",
]


def parse_log_line(line):
    parts = line.split()
    if len(parts) != 5:
        return None
    date, level, user_part, action_part, ms_part = parts
    try:
        return {
            "date": date,
            "level": level,
            "user": user_part.removeprefix("user="),
            "action": action_part.removeprefix("action="),
            "ms": int(ms_part.removeprefix("ms=")),
        }
    except ValueError:
        return None


def average(values):
    count = 0
    total = 0
    for value in values:
        count += 1
        total += value
    return total / count if count else None

parsed = (parse_log_line(line) for line in sample_logs)
clean = (row for row in parsed if row is not None)
errors = (row for row in clean if row["level"] == "ERROR")
error_ms = (row["ms"] for row in errors)

answer = average(error_ms)
print(answer)
assert answer == (870 + 430 + 90) / 3


463.3333333333333


---

## Problem 3 — Find First Matching Item Lazily

Use `next(...)` with a generator expression to return the first transaction above a threshold. Stop scanning immediately after the first match.


In [5]:
def transaction_stream():
    print("creating transaction 1")
    yield {"id": 1, "amount": 12.50}
    print("creating transaction 2")
    yield {"id": 2, "amount": 19.99}
    print("creating transaction 3")
    yield {"id": 3, "amount": 10_000.00}
    print("creating transaction 4")
    yield {"id": 4, "amount": 8.25}


def first_transaction_above(transactions, threshold):
    return next((tx for tx in transactions if tx["amount"] > threshold), None)

match = first_transaction_above(transaction_stream(), 1_000)
print(match)
assert match == {"id": 3, "amount": 10_000.00}


creating transaction 1
creating transaction 2
creating transaction 3
{'id': 3, 'amount': 10000.0}


---

## Problem 4 — Short-Circuit Validation with `all` and `any`

Implement `all_users_valid(users)` and `has_admin(users)` using generator expressions.


In [6]:
users = [
    {"name": "Ana", "age": 31, "role": "member"},
    {"name": "Bob", "age": 28, "role": "admin"},
    {"name": "Chen", "age": 25, "role": "member"},
]


def valid_user(user):
    return (
        isinstance(user.get("name"), str)
        and bool(user["name"].strip())
        and isinstance(user.get("age"), int)
        and user["age"] > 0
    )


def all_users_valid(users):
    return all(valid_user(user) for user in users)


def has_admin(users):
    return any(user.get("role") == "admin" for user in users)

assert all_users_valid(users) is True
assert has_admin(users) is True
assert all_users_valid(users + [{"name": "", "age": 40, "role": "member"}]) is False
print("validation passed")


validation passed


### Short-Circuit Demonstration


In [7]:
def noisy_is_even(number):
    print(f"checking {number}")
    return number % 2 == 0

print("any stops once it finds True:")
print(any(noisy_is_even(n) for n in [1, 3, 5, 8, 10]))

print("all stops once it finds False:")
print(all(noisy_is_even(n) for n in [2, 4, 6, 7, 8]))


any stops once it finds True:
checking 1
checking 3
checking 5
checking 8
True
all stops once it finds False:
checking 2
checking 4
checking 6
checking 7
False


---

## Problem 5 — Nested Generator Late-Binding Bug

Build a lazy multiplication table. First observe the bug, then fix it.

Required output for `start=1`, `stop=4`:

```python
[[1, 2, 3, 4], [2, 4, 6, 8], [3, 6, 9, 12], [4, 8, 12, 16]]
```


In [8]:
def broken_multiplication_table(start, stop):
    return ((i * j for j in range(start, stop + 1)) for i in range(start, stop + 1))

broken_rows = list(broken_multiplication_table(1, 4))
print([list(row) for row in broken_rows])  # usually wrong: every row uses final i


[[4, 8, 12, 16], [4, 8, 12, 16], [4, 8, 12, 16], [4, 8, 12, 16]]


In [9]:
def row_for(i, start, stop):
    return (i * j for j in range(start, stop + 1))


def multiplication_table_lazy_rows(start, stop):
    return (row_for(i, start, stop) for i in range(start, stop + 1))


def multiplication_table_lazy_outer(start, stop):
    return ([i * j for j in range(start, stop + 1)] for i in range(start, stop + 1))

expected = [[1, 2, 3, 4], [2, 4, 6, 8], [3, 6, 9, 12], [4, 8, 12, 16]]
assert [list(row) for row in multiplication_table_lazy_rows(1, 4)] == expected
assert list(multiplication_table_lazy_outer(1, 4)) == expected
print(expected)


[[1, 2, 3, 4], [2, 4, 6, 8], [3, 6, 9, 12], [4, 8, 12, 16]]


---

## Problem 6 — Pascal Triangle as an Infinite Lazy Stream

Create lazy Pascal triangle rows. Then take only the number of rows needed.


In [10]:
def combo(n, k):
    return factorial(n) // (factorial(k) * factorial(n - k))


def pascal_rows_factorial():
    n = 0
    while True:
        yield [combo(n, k) for k in range(n + 1)]
        n += 1

first_six = list(islice(pascal_rows_factorial(), 6))
for row in first_six:
    print(row)
assert first_six[-1] == [1, 5, 10, 10, 5, 1]


[1]
[1, 1]
[1, 2, 1]
[1, 3, 3, 1]
[1, 4, 6, 4, 1]
[1, 5, 10, 10, 5, 1]


In [11]:
def pascal_rows_fast():
    row = [1]
    while True:
        yield row
        row = [1] + [a + b for a, b in zip(row, row[1:])] + [1]

first_ten = list(islice(pascal_rows_fast(), 10))
for row in first_ten:
    print(row)
assert first_ten[9] == [1, 9, 36, 84, 126, 126, 84, 36, 9, 1]


[1]
[1, 1]
[1, 2, 1]
[1, 3, 3, 1]
[1, 4, 6, 4, 1]
[1, 5, 10, 10, 5, 1]
[1, 6, 15, 20, 15, 6, 1]
[1, 7, 21, 35, 35, 21, 7, 1]
[1, 8, 28, 56, 70, 56, 28, 8, 1]
[1, 9, 36, 84, 126, 126, 84, 36, 9, 1]


---

## Problem 7 — Streaming Top-K Largest Values

Return the `k` largest values from a stream without sorting the whole stream.


In [12]:
def top_k_largest(values, k):
    if k <= 0:
        return []
    heap = []
    for value in values:
        if len(heap) < k:
            heapq.heappush(heap, value)
        elif value > heap[0]:
            heapq.heapreplace(heap, value)
    return sorted(heap, reverse=True)

known = (x for x in [5, 1, 100, 3, 99, 2, 50, 101])
assert top_k_largest(known, 3) == [101, 100, 99]

large_stream = (randint(1, 10_000) for _ in range(100_000))
answer = top_k_largest(large_stream, 7)
print(answer)
assert len(answer) == 7
assert answer == sorted(answer, reverse=True)


[10000, 10000, 10000, 10000, 10000, 10000, 10000]


---

## Problem 8 — Memory Comparison: List vs Generator

Compare peak memory usage for summing one million squares.


In [13]:
def peak_memory_bytes(func, *args, **kwargs):
    tracemalloc.stop()
    tracemalloc.clear_traces()
    tracemalloc.start()
    result = func(*args, **kwargs)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, peak


def sum_squares_list(n):
    return sum([i * i for i in range(n)])


def sum_squares_gen(n):
    return sum(i * i for i in range(n))

n = 1_000_000
list_result, list_peak = peak_memory_bytes(sum_squares_list, n)
gen_result, gen_peak = peak_memory_bytes(sum_squares_gen, n)

print(f"list peak memory: {list_peak:,} bytes")
print(f"gen  peak memory: {gen_peak:,} bytes")
assert list_result == gen_result
assert gen_peak < list_peak


list peak memory: 40,449,048 bytes
gen  peak memory: 528 bytes


---

## Problem 9 — Chunked File Processing

Read a file lazily, parse integers, filter even numbers, and sum the first `limit` even values.


In [14]:
def valid_int_or_none(text):
    text = text.strip()
    if not text:
        return None
    try:
        return int(text)
    except ValueError:
        return None


def sum_first_even_numbers_from_file(path, limit):
    with open(path, "r", encoding="utf-8") as file:
        parsed = (valid_int_or_none(line) for line in file)
        numbers = (number for number in parsed if number is not None)
        evens = (number for number in numbers if number % 2 == 0)
        return sum(islice(evens, limit))

with NamedTemporaryFile("w+", encoding="utf-8", delete=False) as tmp:
    path = tmp.name
    tmp.write("\n".join(str(i) for i in range(1, 10_001)))
    tmp.write("\nnot-an-int\n")

result = sum_first_even_numbers_from_file(path, 5)
print(result)
assert result == 2 + 4 + 6 + 8 + 10
Path(path).unlink(missing_ok=True)


30


---

## Problem 10 — Flatten Nested Generators Lazily

Flatten a nested iterable and filter values above a threshold. Show both nested-generator and `chain.from_iterable` solutions.


In [15]:
matrix = ((i * j for j in range(1, 6)) for i in range(1, 6))

solution_a = (value for row in matrix for value in row if value > 10)
print(list(solution_a))

matrix = ((i * j for j in range(1, 6)) for i in range(1, 6))
flattened = chain.from_iterable(matrix)
solution_b = (value for value in flattened if value > 10)
answer = list(solution_b)
print(answer)
assert answer == [12, 15, 12, 16, 20, 15, 20, 25]


[12, 15, 12, 16, 20, 15, 20, 25]
[12, 15, 12, 16, 20, 15, 20, 25]


---

## Problem 11 — Sliding Window Generator

Implement `sliding_window(iterable, size)` for any iterable, including generators.


In [16]:
def sliding_window(iterable, size):
    if size <= 0:
        raise ValueError("size must be positive")
    iterator = iter(iterable)
    window = deque(islice(iterator, size), maxlen=size)
    if len(window) == size:
        yield tuple(window)
    for item in iterator:
        window.append(item)
        yield tuple(window)

assert list(sliding_window([1, 2, 3, 4, 5], 3)) == [(1, 2, 3), (2, 3, 4), (3, 4, 5)]
assert list(sliding_window((x for x in range(4)), 2)) == [(0, 1), (1, 2), (2, 3)]
assert list(sliding_window([1, 2], 3)) == []
print(list(sliding_window([1, 2, 3, 4, 5], 3)))


[(1, 2, 3), (2, 3, 4), (3, 4, 5)]


In [17]:
def moving_average(values, window_size):
    return (sum(window) / window_size for window in sliding_window(values, window_size))

assert list(moving_average([10, 20, 30, 40, 50], 3)) == [20.0, 30.0, 40.0]
print(list(moving_average((x for x in [10, 20, 30, 40, 50]), 3)))


[20.0, 30.0, 40.0]


---

## Problem 12 — Generator Pipeline for CSV-like Orders

Calculate total revenue by country for completed orders only. Skip malformed rows.


In [18]:
orders = [
    {"country": "BG", "status": "complete", "quantity": "2", "unit_price": "10.50"},
    {"country": "BG", "status": "pending", "quantity": "9", "unit_price": "99.00"},
    {"country": "DE", "status": "complete", "quantity": "1", "unit_price": "20.00"},
    {"country": "BG", "status": "complete", "quantity": "3", "unit_price": "7.25"},
    {"country": "FR", "status": "complete", "quantity": "bad", "unit_price": "12.00"},
    {"country": "DE", "status": "complete", "quantity": "4", "unit_price": "5.00"},
]


def cleaned_order(row):
    if row.get("status") != "complete":
        return None
    try:
        return {
            "country": row["country"],
            "revenue": int(row["quantity"]) * float(row["unit_price"]),
        }
    except (KeyError, TypeError, ValueError):
        return None


def revenue_by_country(rows):
    totals = Counter()
    cleaned = (cleaned_order(row) for row in rows)
    valid = (row for row in cleaned if row is not None)
    for row in valid:
        totals[row["country"]] += row["revenue"]
    return dict(totals)

result = revenue_by_country(orders)
print(result)
assert isclose(result["BG"], 2 * 10.50 + 3 * 7.25)
assert isclose(result["DE"], 1 * 20.00 + 4 * 5.00)
assert "FR" not in result


{'BG': 42.75, 'DE': 40.0}


---

## Problem 13 — Reusing a Generator with `tee`

Demonstrate that a generator is consumed once, then use `tee` to split it.


In [19]:
def numbers():
    for i in range(5):
        print(f"yielding {i}")
        yield i

single = numbers()
print("first pass:", list(single))
print("second pass:", list(single))

source = numbers()
a, b = tee(source, 2)
print("a:", list(a))
print("b:", list(b))

# Caution: tee may cache internally if one branch gets far ahead of another.


yielding 0
yielding 1
yielding 2
yielding 3
yielding 4
first pass: [0, 1, 2, 3, 4]
second pass: []
yielding 0
yielding 1
yielding 2
yielding 3
yielding 4
a: [0, 1, 2, 3, 4]
b: [0, 1, 2, 3, 4]


---

## Problem 14 — Infinite Fibonacci Stream

Create an infinite Fibonacci generator and safely limit it with `islice` and `takewhile`.


In [20]:
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

first_12 = list(islice(fibonacci(), 12))
print(first_12)
assert first_12 == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]

below_1000 = list(takewhile(lambda x: x < 1000, fibonacci()))
print(below_1000)
assert below_1000[-1] == 987

even_sum = sum(x for x in takewhile(lambda n: n < 10_000, fibonacci()) if x % 2 == 0)
print(even_sum)
assert even_sum == 3382


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987]
3382


---

## Problem 15 — Debugging Lazy Pipelines with `tap`

Write `tap(iterable, label)` that prints each item as it passes through and yields it unchanged.


In [21]:
def tap(iterable, label="item"):
    for item in iterable:
        print(f"{label}: {item!r}")
        yield item

pipeline = (
    x * 10
    for x in tap((n for n in range(6)), label="original")
    if x % 2 == 0
)

print("Before consumption: nothing has printed yet.")
result = list(pipeline)
print("Result:", result)
assert result == [0, 20, 40]


Before consumption: nothing has printed yet.
original: 0
original: 1
original: 2
original: 3
original: 4
original: 5
Result: [0, 20, 40]


---

## Problem 16 — Reusable Lazy ETL Helpers

Implement `where`, `select`, `take`, and `pipe`, then process numbers lazily.


In [22]:
def where(predicate, iterable):
    return (item for item in iterable if predicate(item))


def select(transform, iterable):
    return (transform(item) for item in iterable)


def take(n, iterable):
    return islice(iterable, n)


def pipe(iterable, *steps):
    result = iterable
    for step in steps:
        result = step(result)
    return result

processed = pipe(
    range(1, 101),
    lambda data: where(lambda x: x % 3 == 0, data),
    lambda data: select(lambda x: x ** 2, data),
    lambda data: where(lambda x: x < 2_000, data),
    lambda data: take(10, data),
)

answer = list(processed)
print(answer)
assert answer == [9, 36, 81, 144, 225, 324, 441, 576, 729, 900]


[9, 36, 81, 144, 225, 324, 441, 576, 729, 900]


---

## Problem 17 — Streaming Join

Join a large event stream to a smaller user table. Materialize only the smaller lookup.


In [23]:
events = (
    {"event_id": 1, "user_id": 10, "action": "login"},
    {"event_id": 2, "user_id": 20, "action": "download"},
    {"event_id": 3, "user_id": 999, "action": "unknown-user"},
    {"event_id": 4, "user_id": 10, "action": "logout"},
)

users = [
    {"user_id": 10, "name": "Ana", "plan": "pro"},
    {"user_id": 20, "name": "Bob", "plan": "free"},
]


def join_events_with_users(events, users):
    user_lookup = {user["user_id"]: user for user in users}
    for event in events:
        user = user_lookup.get(event.get("user_id"))
        if user is None:
            continue
        yield {
            **event,
            "user_name": user["name"],
            "user_plan": user["plan"],
        }

joined_rows = list(join_events_with_users(events, users))
print(joined_rows)
assert joined_rows == [
    {"event_id": 1, "user_id": 10, "action": "login", "user_name": "Ana", "user_plan": "pro"},
    {"event_id": 2, "user_id": 20, "action": "download", "user_name": "Bob", "user_plan": "free"},
    {"event_id": 4, "user_id": 10, "action": "logout", "user_name": "Ana", "user_plan": "pro"},
]


[{'event_id': 1, 'user_id': 10, 'action': 'login', 'user_name': 'Ana', 'user_plan': 'pro'}, {'event_id': 2, 'user_id': 20, 'action': 'download', 'user_name': 'Bob', 'user_plan': 'free'}, {'event_id': 4, 'user_id': 10, 'action': 'logout', 'user_name': 'Ana', 'user_plan': 'pro'}]


---

## Problem 18 — Refactor an Over-Complex Generator Expression

Refactor this expression into readable helper functions while keeping lazy behavior:

```python
(x["score"] * 1.1 for x in rows if x.get("active") and x.get("score") is not None and x["score"] >= 70)
```


In [24]:
rows = [
    {"name": "Ana", "active": True, "score": 80},
    {"name": "Bob", "active": False, "score": 95},
    {"name": "Chen", "active": True, "score": None},
    {"name": "Dina", "active": True, "score": 68},
    {"name": "Eli", "active": True, "score": 100},
]


def is_eligible(row):
    score = row.get("score")
    return row.get("active") is True and score is not None and score >= 70


def adjusted_score(row):
    return row["score"] * 1.1

scores = (adjusted_score(row) for row in rows if is_eligible(row))
answer = list(scores)
print(answer)
assert answer == [88.0, 110.00000000000001]


[88.0, 110.00000000000001]


## Final Notes

Generator expressions are not just shorter syntax. They are a design tool for lazy, memory-conscious, composable programs.

When solving advanced problems, always ask:

1. Do I need all values at once?
2. Will I consume this result more than once?
3. Can I stop early?
4. Is my generator expression still readable?
5. Would a small generator function be clearer?
